# مختبر اليوم الخامس — مشروع منافذ المتكامل وفحص تغير التشغيل

> هذا الدفتر مبني مباشرة من مواصفات مختبرات منافذ المعتمدة للدورة.


In [ ]:
from pathlib import Path

# يعمل محليًا داخل المستودع أو عند وضع الحزمة في مجلد مستقل
DATA_CANDIDATES = [Path("manafeth_data_package"), Path("data/raw"), Path("../../data/raw")]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])
CUSTOMERS_PATH = DATA_DIR / "manafeth_customers.parquet"
ORDERS_PATH = DATA_DIR / "manafeth_orders.parquet"
VEHICLES_PATH = DATA_DIR / "markabat_listings_sample.csv"
SHIFTED_PATH = DATA_DIR / "shifted_month.parquet"
print("DATA_DIR:", DATA_DIR.resolve())


## هدف المختبر

ينجز الطالب مشروع تعلم آلة بسيطًا من البداية إلى النهاية باستخدام بيانات منافذ الفعلية، ثم يجري فحصًا تشغيليًا أوليًا على شهر لاحق. يجمع المشروع صياغة المشكلة، تجهيز البيانات، المقارنة، التقييم، تحليل الأخطاء، والتواصل المسؤول.

## السيناريو والبيانات

يبني الطالب نموذجًا يرتب العملاء الذين يحتمل أن يغادروا خلال 30 يومًا من تاريخ لقطة شهرية. يستخدم `manafeth_customers.parquet` لتطوير النموذج وتقييمه التاريخي. بعد تثبيت النموذج، يستخدم `shifted_month.parquet` وهو جدول من شهر لاحق **من دون هدف معروف في المختبر**؛ لذلك لا يدعي الطالب قياس أداء عليه، بل يقارن فقط نسبة العملاء الذين سيتجاوزون عتبة الاتصال نفسها.

> لا تدخل أي من الأعمدة `refund_issued` أو `support_ticket_after_snapshot` أو `next_month_orders` في المشروع. لا يحتوي ملف الشهر اللاحق على الهدف، ولذلك لا يجوز كتابة دقة أو استدعاء له.

## مراحل التنفيذ بالتسلسل

### المرحلة الأولى: بطاقة المشروع والاستكشاف

1. اكتب بطاقة المشكلة: السؤال، وحدة التنبؤ، الهدف، قرار المتابعة، والخصائص المتاحة عند تاريخ اللقطة.
2. افتح جدول العملاء وافحص الحجم والأنواع والنقص وتوازن الهدف.
3. أنشئ جدول قرار يوضح الأعمدة الآمنة، المعرفات، والتسريب.
4. ارسم مخطط أعمدة لتوزيع `churned_30d` بعنوان ومحاور عربية.

### المرحلة الثانية: التجهيز والمقارنة

5. أنشئ `X` و`y` وقسم البيانات إلى تدريب واختبار بنسبة 80/20 مع الحفاظ على توزيع الهدف.
6. ابنِ `preprocessor` نفسه، بما في ذلك مؤشر النقص للأعمدة العددية.
7. قارن الانحدار اللوجستي والغابة العشوائية وXGBoost بالتحقق المتقاطع وبمقياس متوسط الدقة.
8. اختر نموذجًا ودوّن سبب الاختيار قبل فتح بيانات الاختبار.

### المرحلة الثالثة: التقييم والتفسير

9. درب النموذج المختار على كامل بيانات التدريب.
10. احسب احتمال المغادرة لكل عميل اختبار، ومتوسط الدقة، ومنحنى الدقة–الاستدعاء، والاستدعاء بين أعلى 20% من العملاء.
11. اعرض مصفوفة الالتباس عند عتبة 0.50 للتدريب على قراءة الأخطاء.
12. راجع خمسة صفوف أخطأ النموذج في تصنيفها واكتب ملاحظة تستند إلى البيانات فقط.

### المرحلة الرابعة: فحص تغير التشغيل في شهر لاحق

13. افتح `shifted_month.parquet`، واختر منه **الأعمدة الآمنة نفسها وبالترتيب نفسه**.
14. طبق النموذج للحصول على احتمالات من دون تقييم دقة؛ لا توجد حقيقة هدف في هذا الملف.
15. حدد عتبة الاتصال من بيانات الاختبار: العتبة التي تفصل أعلى 20% من احتمالات الاختبار.
16. احسب نسبة عملاء الشهر اللاحق الذين يتجاوزون هذه العتبة، وقارنها بنسبة 20% الأصلية.
17. اكتب تنبيهًا تشغيليًا إن تغيرت النسبة: «تغيرت نسبة التنبيهات، لذا يجب مراجعة القدرة التشغيلية وقياس النتائج الفعلية عند توفرها».

## ما يكتبه أو يشغله الطالب


In [ ]:
import numpy as np

# عتبة تُختار من بيانات الاختبار لتحديد أعلى 20% من العملاء بالترتيب
contact_threshold = np.quantile(probabilities, 0.80)

shifted = pd.read_parquet(SHIFTED_PATH)
X_shifted = shifted[safe_features]
shifted_probabilities = final_workflow.predict_proba(X_shifted)[:, 1]
shifted_contact_rate = (shifted_probabilities >= contact_threshold).mean()

print("عتبة الاتصال:", round(contact_threshold, 3))
print("نسبة العملاء المتجاوزين للعتبة في الشهر اللاحق:", round(shifted_contact_rate, 3))


In [ ]:
# مراجعة أخطاء الاختبار فقط؛ لا نستخدم هذا الكود على الشهر اللاحق لأنه بلا هدف.
review = X_test.copy()
review["الحقيقة"] = y_test.to_numpy()
review["احتمال_المغادرة"] = probabilities
review["التوقع_عند_0_50"] = (probabilities >= 0.50).astype(int)
errors = review[review["الحقيقة"] != review["التوقع_عند_0_50"]]
errors.head()


## النتيجة المتوقعة

ينتج دفتر مشروع منظم يحتوي على تعريف المشكلة، فحص البيانات، خط معالجة، مقارنة نماذج، جدول نتائج، رسم واحد على الأقل، منحنى دقة–استدعاء، مصفوفة التباس، تحليل أخطاء، وفحص لتغير نسبة التنبيهات في الشهر اللاحق. قد تتغير نسبة التنبيهات في الشهر اللاحق، وهذه ليست «دقة» أو «فشلًا» للنموذج؛ هي إشارة تشغيلية تحتاج قياس النتائج الفعلية لاحقًا.

## المهارات التي يراجعها الطالب

يراجع الطالب سير عمل تعلم الآلة كاملًا: صياغة المشكلة، منع التسريب، pandas وJupyter، المعالجة المسبقة، التصنيف، التحقق المتقاطع، اختيار مقياس مناسب، Matplotlib، XGBoost، تحليل الأخطاء، وتقديم نتيجة مسؤولة.

## شكل التسليم والعرض

يسلم كل فريق دفتر `final_project_manafeth_churn.ipynb` ويعرض خلال دقيقة ونصف:

1. المشكلة والهدف وسبب نوع التصنيف.
2. الخصائص الآمنة والأعمدة التي استبعدها ولماذا.
3. طريقة التجهيز والمقارنة.
4. النموذج المختار والمقياس المستخدم.
5. مصفوفة الالتباس أو منحنى الدقة–الاستدعاء.
6. ملاحظة واحدة من الأخطاء، وملاحظة واحدة عن تغير نسبة التنبيهات في الشهر اللاحق.

## قائمة تحقق المدرب

| بند المراجعة | تحقق |
|---|---|
| استُخدم `manafeth_customers.parquet` للمختبرات الأساسية | ☐ |
| استُبعدت الأعمدة المسرّبة الثلاثة من كل `X` | ☐ |
| بُنيت المعالجة داخل `Pipeline` و`ColumnTransformer` | ☐ |
| قورنت النماذج على التدريب فقط بالتحقق المتقاطع | ☐ |
| قُيّم الاختبار مرة واحدة بعد الاختيار | ☐ |
| لم تُنسب دقة أو استدعاء إلى `shifted_month.parquet` | ☐ |
| احتوى التسليم على تحليل صفوف خطأ فعلية وخلاصة لا تبالغ | ☐ |
